## CELL 1 — Install and Imports



In [2]:
!pip uninstall -y transformers tokenizers autoawq awq optimum auto-gptq gptqmodel

!pip install -q "transformers==4.51.3"
!pip install -q "tokenizers>=0.21.0,<0.22.0"
!pip install -q "autoawq==0.2.9"
!pip install -q --upgrade accelerate safetensors sentencepiece einops

Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
Found existing installation: tokenizers 0.22.2
Uninstalling tokenizers-0.22.2:
  Successfully uninstalled tokenizers-0.22.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 82.6 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 37.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 74.5 MB/s eta 0:00:00:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.3/74.3 kB 2.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 8.1 MB/s eta 0:00:00:00:01


In [3]:
!pip install -q evaluate sacremoses sacrebleu textstat


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 21.5 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.1/177.1 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 63.9 MB/s eta 0:00:00


In [4]:
# ============================================================
# Suppress non-critical warnings and noisy backend logs
# Put this cell before importing torch / tensorflow / transformers
# ============================================================

import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"
os.environ["PYTHONWARNINGS"] = "ignore"

import warnings
import logging

warnings.simplefilter("ignore")
warnings.filterwarnings("ignore")

logging.disable(logging.WARNING)

try:
    import absl.logging
    absl.logging.set_verbosity(absl.logging.ERROR)
    absl.logging.set_stderrthreshold("error")
except Exception:
    pass

try:
    from transformers.utils import logging as hf_logging
    hf_logging.set_verbosity_error()
except Exception:
    pass

print("Warning suppression cell executed.")

Warning suppression cell executed.


In [ ]:
import os
import signal

print("Restarting kernel now...")
os.kill(os.getpid(), signal.SIGTERM)

In [1]:
import transformers
import tokenizers
import torch
import awq

print("Transformers:", transformers.__version__)
print("Tokenizers:", tokenizers.__version__)
print("Torch:", torch.__version__)
print("AWQ:", awq.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

/usr/local/lib/python3.12/dist-packages/awq/__init__.py:21: DeprecationWarning: 
I have left this message as the final dev message to help you transition.

Important Notice:
- AutoAWQ is officially deprecated and will no longer be maintained.
- The last tested configuration used Torch 2.6.0 and Transformers 4.51.3.
- If future versions of Transformers break AutoAWQ compatibility, please report the issue to the Transformers project.

Alternative:
- AutoAWQ has been adopted by the vLLM Project: https://github.com/vllm-project/llm-compressor

For further inquiries, feel free to reach out:
- X: https://x.com/casper_hansen_
- LinkedIn: https://www.linkedin.com/in/casper-hansen-804005170/

  warnings.warn(_FINAL_DEV_MESSAGE, category=DeprecationWarning, stacklevel=1)
2026-05-07 11:38:20.173195: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:177815

Transformers: 4.51.3
Tokenizers: 0.21.4
Torch: 2.10.0+cu128
AWQ: 0.2.9
CUDA available: True
GPU: Tesla T4


/usr/local/lib/python3.12/dist-packages/torch/jit/_script.py:1480: DeprecationWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [2]:
INSTALL_OR_UPGRADE_PACKAGES = False

if INSTALL_OR_UPGRADE_PACKAGES:
    import sys
    import subprocess

    packages = [
        "transformers>=4.51.0",
        "accelerate",
        "bitsandbytes",
        "sentencepiece",
        "evaluate",
        "sacrebleu",
        "sacremoses",
        "textstat",
        "tqdm",
    ]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", *packages])

import os
import re
import gc
import json
import time
import random
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


Torch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


## CELL 2 — Configuration




In [3]:
from pathlib import Path
import random
import numpy as np
import pandas as pd
import torch

In [4]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


TRAIN_PATH = Path("/kaggle/input/datasets/raneemrabi3/cochraneauto-sents/cochraneauto_sents_train.csv")
VALID_PATH = Path("/kaggle/input/datasets/raneemrabi3/cochraneauto-sents/cochraneauto_sents_val.csv")
TEST_INTERNAL_PATH = Path("/kaggle/input/datasets/raneemrabi3/cochraneauto-sents/cochraneauto_sents_test.csv")
OFFICIAL_TEST_JSON = Path("/kaggle/input/datasets/raneemrabi3/testjson/simpletext25_task11_test (1).json")
MEDSIMPLIFY_PATH = Path("/kaggle/input/datasets/raneemrabi3/medsimplify/MedSimplify.csv")


PROJECT_DIR = Path("/kaggle/working/simpletext_qwen25_14b_clean")
OUTPUT_DIR = PROJECT_DIR / "outputs"
RUN_DIR = OUTPUT_DIR / "runs"

PROJECT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR.mkdir(parents=True, exist_ok=True)


MODEL_PATH_OVERRIDE = Path(
    "/kaggle/input/models/qwen-lm/qwen2.5/transformers/14b-instruct-awq/1"
)

MODEL_KEYWORDS = ["qwen", "2.5", "14b", "instruct", "awq"]
REJECT_MODEL_KEYWORDS = ["base", "gptq"]

USE_4BIT = False
TRUST_REMOTE_CODE = True
LOCAL_FILES_ONLY = True


MAX_INPUT_TOKENS = 768
MAX_NEW_TOKENS = 128
PLANNER_MAX_TOKENS = 180

PLANNER_TEMPERATURE = 0.0
EXECUTOR_TEMPERATURE = 0.2

TOP_P = 0.9
GENERATION_MAX_RETRIES = 2
REQUEST_SLEEP_SEC = 0.0
EMPTY_CACHE_EVERY_N_CALLS = 10


TEMPERATURE = EXECUTOR_TEMPERATURE


SAMPLE_SIZE = 100
SAMPLE_SPLIT_NAME = "valid_100_stratified_qwen25_14b_awq_quick_v1"
STRATIFY_BY_LABEL_AND_LENGTH = True


VALID_RUN_SIZE = 500
INTERNAL_TEST_RUN_SIZE = 500
VALID_RUN_SPLIT_NAME = "valid_900_stratified_qwen25_14b_awq_final_v1"
INTERNAL_TEST_RUN_SPLIT_NAME = "internal_test_800_stratified_qwen25_14b_awq_final_v1"


RESET_BATCH_RUNS = False


RAG_MAX_TERMS = 3

print("PROJECT_DIR:", PROJECT_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("MODEL_PATH_OVERRIDE:", MODEL_PATH_OVERRIDE)
print("MODEL exists:", MODEL_PATH_OVERRIDE.exists())

print("TRAIN exists:", TRAIN_PATH.exists())
print("VALID exists:", VALID_PATH.exists())
print("TEST_INTERNAL exists:", TEST_INTERNAL_PATH.exists())
print("MEDSIMPLIFY exists:", MEDSIMPLIFY_PATH.exists())

print("MAX_INPUT_TOKENS:", MAX_INPUT_TOKENS)
print("MAX_NEW_TOKENS:", MAX_NEW_TOKENS)
print("PLANNER_MAX_TOKENS:", PLANNER_MAX_TOKENS)
print("PLANNER_TEMPERATURE:", PLANNER_TEMPERATURE)
print("EXECUTOR_TEMPERATURE:", EXECUTOR_TEMPERATURE)
print("VALID_RUN_SIZE:", VALID_RUN_SIZE)
print("INTERNAL_TEST_RUN_SIZE:", INTERNAL_TEST_RUN_SIZE)


PROJECT_DIR: /kaggle/working/simpletext_qwen25_14b_clean
OUTPUT_DIR: /kaggle/working/simpletext_qwen25_14b_clean/outputs
MODEL_PATH_OVERRIDE: /kaggle/input/models/qwen-lm/qwen2.5/transformers/14b-instruct-awq/1
MODEL exists: True
TRAIN exists: True
VALID exists: True
TEST_INTERNAL exists: True
MEDSIMPLIFY exists: True
MAX_INPUT_TOKENS: 768
MAX_NEW_TOKENS: 128
PLANNER_MAX_TOKENS: 180
PLANNER_TEMPERATURE: 0.0
EXECUTOR_TEMPERATURE: 0.2
VALID_RUN_SIZE: 500
INTERNAL_TEST_RUN_SIZE: 500


## CELL 3 — Basic Text Utilities

In [5]:
def normalize_text(text: object) -> str:
    if pd.isna(text):
        return ""
    text = str(text).replace("\n", " ").replace("\r", " ")
    return re.sub(r"\s+", " ", text).strip()


def normalize_term_text(text: object) -> str:
    text = str(text).strip().lower()
    return re.sub(r"\s+", " ", text)


def safe_word_count(text: object) -> int:
    return len(normalize_text(text).split())


def safe_fkgl(text: object) -> float:
    text = normalize_text(text)
    if not text:
        return np.nan
    try:
        import textstat
        return float(textstat.flesch_kincaid_grade(text))
    except Exception:
        return np.nan


def ensure_row_id(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if "row_id" not in df.columns:
        df.insert(0, "row_id", np.arange(len(df)))
    return df


def require_columns(df: pd.DataFrame, columns: List[str], name: str) -> None:
    missing = [c for c in columns if c not in df.columns]
    if missing:
        raise ValueError(f"{name} is missing required columns: {missing}")

## CELL 4 — Local Qwen Model Setup



In [6]:
import os
from pathlib import Path
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM


def list_kaggle_input_dirs(max_lines: int = 120) -> None:
    printed = 0
    for dirname, _, _ in os.walk("/kaggle/input"):
        print(dirname)
        printed += 1
        if printed >= max_lines:
            print("... output truncated")
            break


def find_local_model_path(
    base_dir="/kaggle/input",
    keywords=None,
    reject_keywords=None,
    override_path=None,
):
    if override_path is not None:
        override_path = Path(override_path)

        if not override_path.exists():
            raise FileNotFoundError(f"MODEL_PATH_OVERRIDE does not exist: {override_path}")

        if not (override_path / "config.json").exists():
            raise FileNotFoundError(f"config.json not found inside: {override_path}")

        return override_path

    keywords = keywords or []
    reject_keywords = reject_keywords or []
    candidates = []

    for root, _, files in os.walk(base_dir):
        root_path = Path(root)
        root_lower = str(root_path).lower()

        if "config.json" not in files:
            continue

        if not all(keyword.lower() in root_lower for keyword in keywords):
            continue

        if any(keyword.lower() in root_lower for keyword in reject_keywords):
            continue

        candidates.append(root_path)

    if not candidates:
        raise FileNotFoundError(
            f"No model path found with keywords={keywords} and reject_keywords={reject_keywords}"
        )

    candidates = sorted(candidates, key=lambda p: len(str(p)))
    return candidates[0]


MODEL_PATH = find_local_model_path(
    keywords=MODEL_KEYWORDS,
    reject_keywords=REJECT_MODEL_KEYWORDS,
    override_path=MODEL_PATH_OVERRIDE,
)

print("Using MODEL_PATH:", MODEL_PATH)
print("config exists:", (MODEL_PATH / "config.json").exists())


tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH,
    trust_remote_code=TRUST_REMOTE_CODE,
    local_files_only=LOCAL_FILES_ONLY,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    device_map="auto",
    trust_remote_code=TRUST_REMOTE_CODE,
    local_files_only=LOCAL_FILES_ONLY,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
)

model.eval()

print("Tokenizer loaded:", type(tokenizer).__name__)
print("Model loaded:", type(model).__name__)
print("Device map:", getattr(model, "hf_device_map", "not available"))

Using MODEL_PATH: /kaggle/input/models/qwen-lm/qwen2.5/transformers/14b-instruct-awq/1
config exists: True


Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Tokenizer loaded: Qwen2TokenizerFast
Model loaded: Qwen2ForCausalLM
Device map: {'model.embed_tokens': 0, 'model.layers.0': 0, 'model.layers.1': 0, 'model.layers.2': 0, 'model.layers.3': 0, 'model.layers.4': 0, 'model.layers.5': 0, 'model.layers.6': 0, 'model.layers.7': 0, 'model.layers.8': 0, 'model.layers.9': 0, 'model.layers.10': 0, 'model.layers.11': 0, 'model.layers.12': 0, 'model.layers.13': 0, 'model.layers.14': 1, 'model.layers.15': 1, 'model.layers.16': 1, 'model.layers.17': 1, 'model.layers.18': 1, 'model.layers.19': 1, 'model.layers.20': 1, 'model.layers.21': 1, 'model.layers.22': 1, 'model.layers.23': 1, 'model.layers.24': 1, 'model.layers.25': 1, 'model.layers.26': 1, 'model.layers.27': 1, 'model.layers.28': 1, 'model.layers.29': 1, 'model.layers.30': 1, 'model.layers.31': 1, 'model.layers.32': 1, 'model.layers.33': 1, 'model.layers.34': 1, 'model.layers.35': 1, 'model.layers.36': 1, 'model.layers.37': 1, 'model.layers.38': 1, 'model.layers.39': 1, 'model.layers.40': 1, 'm

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## CELL 5 — Local Chat Generation with Retry



In [7]:
GENERATION_CALL_COUNT = 0


def first_model_device():
    """Return a safe input device for models loaded with or without device_map='auto'."""
    try:
        return next(model.parameters()).device
    except StopIteration:
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def build_chat_text(messages: List[dict]) -> str:
    """Build a model-specific chat prompt while keeping compatibility with tokenizers that do not support enable_thinking."""
    try:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
    except TypeError:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )


def clean_model_output(text: str) -> str:
    """Remove possible reasoning tags and special tokens from generated text."""
    text = str(text or "")
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL | re.IGNORECASE)
    text = text.replace("<|im_end|>", "").replace("<|endoftext|>", "")
    return normalize_text(text)


import time


def local_chat_call(
    prompt: str,
    system_message: str = "You are a careful text simplification assistant.",
    max_tokens: int = MAX_NEW_TOKENS,
    temperature: float = EXECUTOR_TEMPERATURE,
    retries: int = GENERATION_MAX_RETRIES,
) -> str:
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt},
    ]

    text = build_chat_text(messages)

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_INPUT_TOKENS,
    ).to(first_model_device())

    do_sample = temperature is not None and temperature > 0.0

    generation_kwargs = {
        "max_new_tokens": max_tokens,
        "do_sample": do_sample,
        "pad_token_id": tokenizer.eos_token_id,
    }

    if do_sample:
        generation_kwargs["temperature"] = temperature
        generation_kwargs["top_p"] = TOP_P

    last_error = None

    for attempt in range(retries):
        try:
            with torch.no_grad():
                output_ids = model.generate(
                    **inputs,
                    **generation_kwargs,
                )

            generated_ids = output_ids[0][inputs["input_ids"].shape[-1]:]
            decoded = tokenizer.decode(
                generated_ids,
                skip_special_tokens=True,
            )

            return clean_model_output(decoded)

        except Exception as exc:
            last_error = exc
            print(f"Generation error on attempt {attempt + 1}/{retries}: {exc}")

            if torch.cuda.is_available():
                torch.cuda.empty_cache()

            if REQUEST_SLEEP_SEC > 0:
                time.sleep(REQUEST_SLEEP_SEC)

    raise RuntimeError(f"Generation failed after {retries} retries: {last_error}")


def smoke_test_local_model() -> str:
    prompt = 'Return exactly this JSON: {"ok": true}'
    output = local_chat_call(
        prompt=prompt,
        system_message="Return only valid JSON.",
        max_tokens=30,
        temperature=0.0,
    )
    print("Model response:", output)
    return output


smoke_test_local_model()


/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:653: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


Model response: {"ok": true}


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


'{"ok": true}'

## CELL 6 — Load Data

In [8]:
train_df = ensure_row_id(pd.read_csv(TRAIN_PATH))
valid_df = ensure_row_id(pd.read_csv(VALID_PATH))
test_internal_df = ensure_row_id(pd.read_csv(TEST_INTERNAL_PATH))

require_columns(train_df, ["complex"], "train_df")
require_columns(valid_df, ["complex"], "valid_df")
require_columns(test_internal_df, ["complex"], "test_internal_df")

for df in [train_df, valid_df, test_internal_df]:
    df["complex"] = df["complex"].apply(normalize_text)
    if "simple" in df.columns:
        df["simple"] = df["simple"].apply(normalize_text)
    if "label" in df.columns:
        df["label"] = df["label"].fillna("unknown").astype(str)

print("Train shape:", train_df.shape)
print("Valid shape:", valid_df.shape)
print("Internal test shape:", test_internal_df.shape)

display(train_df.head(3))

Train shape: (11510, 11)
Valid shape: (1697, 11)
Internal test shape: (1512, 11)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,row_id,pair_id,para_id,sent_id,complex,label,simple,simp_sent_id,doc_pos,doc_quint,doc_len
0,0,CD012936,0,0,Three studies (146 participants) met our selec...,rephrase,['Three studies involving 146 participants wer...,0,0.090909,1,11
1,1,CD012936,0,1,"Two studies compared multidisciplinary, fast-t...",delete,[],1,0.181818,1,11
2,2,CD012936,0,2,Two were RCTs with parallel design (total 94 p...,delete,[],1,0.272727,2,11


In [9]:
def token_length(text, tokenizer):
    return len(tokenizer.encode(str(text), add_special_tokens=False))

for name, df in [("train", train_df), ("valid", valid_df)]:
    lengths = df["complex"].apply(lambda x: token_length(x, tokenizer))

    print(f"\n{name.upper()} complex token lengths")
    print("count:", len(lengths))
    print("mean :", int(lengths.mean()))
    print("p50  :", int(lengths.quantile(0.50)))
    print("p90  :", int(lengths.quantile(0.90)))
    print("p95  :", int(lengths.quantile(0.95)))
    print("p99  :", int(lengths.quantile(0.99)))
    print("max  :", int(lengths.max()))


TRAIN complex token lengths
count: 11510
mean : 38
p50  : 28
p90  : 74
p95  : 105
p99  : 188
max  : 557

VALID complex token lengths
count: 1697
mean : 37
p50  : 28
p90  : 73
p95  : 99
p99  : 158
max  : 463


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [10]:
for name, df in [("train", train_df), ("valid", valid_df)]:
    if "simple" in df.columns:
        simple_lengths = df["simple"].apply(lambda x: token_length(x, tokenizer))

        print(f"\n{name.upper()} simple token lengths")
        print("mean :", int(simple_lengths.mean()))
        print("p50  :", int(simple_lengths.quantile(0.50)))
        print("p90  :", int(simple_lengths.quantile(0.90)))
        print("p95  :", int(simple_lengths.quantile(0.95)))
        print("p99  :", int(simple_lengths.quantile(0.99)))
        print("max  :", int(simple_lengths.max()))


TRAIN simple token lengths
mean : 20
p50  : 18
p90  : 46
p95  : 57
p99  : 85
max  : 178

VALID simple token lengths
mean : 19
p50  : 18
p90  : 45
p95  : 54
p99  : 94
max  : 223


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## CELL 7 — MedSimplify Glossary and Terminology Support


In [9]:
med_lookup: Dict[str, str] = {}

if MEDSIMPLIFY_PATH.exists():
    medsimplify_df = pd.read_csv(MEDSIMPLIFY_PATH)
    if not {"word", "definition"}.issubset(set(medsimplify_df.columns)):
        raise ValueError("MedSimplify file must contain columns: word, definition")

    for _, row in medsimplify_df.iterrows():
        term = normalize_term_text(row["word"])
        definition = normalize_text(row["definition"])
        if term and definition:
            med_lookup[term] = definition

print("MedSimplify terms loaded:", len(med_lookup))

RAG_STOP_TERMS = {
    "however", "therefore", "thus", "but", "and", "or", "although",
    "capacity", "measure", "difference", "evidence", "intervention",
    "result", "results", "change", "changes", "risk", "group",
    "groups", "study", "studies", "outcome", "outcomes",
    "evaluate", "effect", "effects", "reduce", "increase",
    "significant", "conclude", "benefit", "oral", "treatment",
    "treatments", "show", "report", "compare", "find", "found",
    "available", "review", "determine", "observe", "detect",
    "mouth", "potential", "moderate", "pregnancy", "regarding",
    "adverse", "complete", "follow-up", "long-term",
    "previous", "individual",
}

# These terms are common in scientific reporting but should not trigger medical substitution.
RAG_STOP_TERMS.update({
    "interval", "confidence interval", "ratio", "odds ratio", "rate", "rates",
    "quality", "trial", "trials", "participant", "participants",
    "intervention", "interventions", "comparison", "comparisons",
    "certainty", "low-certainty", "moderate-certainty", "high-certainty",
    "chance", "chances", "following", "between",
})

WEAK_SINGLE_TERMS = {
    "disease", "therapy", "reaction", "control", "dose", "symptoms",
    "screening", "incidence", "outcome", "outcomes", "pressure",
    "heart", "blood", "pain", "function", "management", "care",
    "chronic", "acute", "cerebral", "pulmonary", "renal",
    "cardiovascular", "medication", "inflammation", "rehabilitation",
    "physician", "biopsy", "fibrillation", "infarction",
}

# Preferred plain-language equivalents used to guide the LLM substitution prompt.
# In the current pipeline, substitution is performed by the model, not by mechanical regex replacement.
SAFE_TERM_REPLACEMENTS = {
    "myocardial infarction": "heart attack",
    "cerebrovascular accident": "stroke",
    "atrial fibrillation": "irregular heartbeat",
    "hypertension": "high blood pressure",
    "hypotension": "low blood pressure",
    "tachycardia": "rapid heart rate",
    "bradycardia": "slow heart rate",
    "dyspnea": "shortness of breath",
    "oedema": "swelling",
    "edema": "swelling",
    "analgesic": "pain-relieving medicine",
    "analgesics": "pain-relieving medicines",
    "glucocorticoid": "steroid medicine",
    "glucocorticoids": "steroid medicines",
    "corticosteroid": "steroid medicine",
    "corticosteroids": "steroid medicines",
    "placebo": "inactive treatment",
    "adverse event": "harmful side effect",
    "adverse events": "harmful side effects",
}


def is_useful_term(term: str, definition: str) -> bool:
    term = normalize_term_text(term)
    definition = normalize_text(definition)

    if not term:
        return False
    if term in RAG_STOP_TERMS:
        return False
    if term in SAFE_TERM_REPLACEMENTS:
        return True
    if not definition:
        return False
    if len(term.split()) == 1 and len(term) <= 3:
        return False
    if len(term.split()) == 1 and term in WEAK_SINGLE_TERMS:
        return False
    if len(definition) < 5:
        return False

    return True


# Search both MedSimplify terms and the curated safe replacement terms.
all_terms = sorted(set(med_lookup.keys()) | set(SAFE_TERM_REPLACEMENTS.keys()), key=len, reverse=True)
multi_word_terms = [t for t in all_terms if len(t.split()) >= 2]
single_word_terms = [t for t in all_terms if len(t.split()) == 1]


def _match_terms_from_list(
    sentence_lower: str,
    terms: List[str],
    found: Dict[str, str],
    occupied_spans: List[Tuple[int, int]],
    max_terms: int,
) -> None:
    for term in terms:
        if len(found) >= max_terms:
            break

        term = normalize_term_text(term)
        definition = med_lookup.get(term, SAFE_TERM_REPLACEMENTS.get(term, ""))

        if not is_useful_term(term, definition):
            continue

        pattern = r"\b" + re.escape(term) + r"\b"

        for match in re.finditer(pattern, sentence_lower):
            start, end = match.span()
            overlaps = any(not (end <= s or start >= e) for s, e in occupied_spans)

            if overlaps:
                continue

            found[term] = definition
            occupied_spans.append((start, end))
            break


def extract_terms(sentence: str, max_terms: int = RAG_MAX_TERMS) -> Dict[str, str]:
    sentence_lower = normalize_text(sentence).lower()
    found: Dict[str, str] = {}
    occupied_spans: List[Tuple[int, int]] = []

    _match_terms_from_list(sentence_lower, multi_word_terms, found, occupied_spans, max_terms)
    _match_terms_from_list(sentence_lower, single_word_terms, found, occupied_spans, max_terms)

    return found


def apply_safe_substitutions(sentence: str, found_terms: Dict[str, str]) -> Tuple[str, Dict[str, str]]:
    """
    Legacy helper kept for compatibility.

    The active pipeline now performs substitution through run_substitute(), using
    the model and the terminology context, then applies the selected strategy.
    This function is not used by the current planner, executor, or interface.
    """
    sentence = normalize_text(sentence)

    if not sentence or not found_terms:
        return sentence, {}

    substituted_sentence = sentence
    applied_substitutions: Dict[str, str] = {}

    terms_to_replace = [
        term for term in found_terms.keys()
        if normalize_term_text(term) in SAFE_TERM_REPLACEMENTS
    ]
    terms_to_replace = sorted(set(terms_to_replace), key=len, reverse=True)

    for term in terms_to_replace:
        normalized_term = normalize_term_text(term)
        replacement = SAFE_TERM_REPLACEMENTS[normalized_term]
        pattern = r"\b" + re.escape(normalized_term) + r"\b"

        new_sentence, count = re.subn(
            pattern,
            replacement,
            substituted_sentence,
            flags=re.IGNORECASE,
        )

        if count > 0:
            substituted_sentence = normalize_text(new_sentence)
            applied_substitutions[normalized_term] = replacement

    return substituted_sentence, applied_substitutions


def build_rag_context(
    found_terms: Dict[str, str],
    applied_substitutions: Optional[Dict[str, str]] = None,
) -> str:
    """
    Build glossary-based terminology hints. This is terminology support,
    not full document retrieval.
    """
    applied_substitutions = applied_substitutions or {}

    if not found_terms and not applied_substitutions:
        return ""

    lines = ["Useful terminology information:"]

    if applied_substitutions:
        lines.append("Safe substitutions already applied:")
        for term, replacement in applied_substitutions.items():
            lines.append(f"- {term} -> {replacement}")

    remaining_terms = {
        term: definition
        for term, definition in found_terms.items()
        if term not in applied_substitutions and term not in RAG_STOP_TERMS
    }

    if remaining_terms:
        lines.append("Additional glossary hints:")
        for term, definition in remaining_terms.items():
            replacement = SAFE_TERM_REPLACEMENTS.get(term)
            if replacement:
                lines.append(f"- {term}: suggested plain wording = {replacement}")
            else:
                lines.append(f"- {term}: {definition}")

    return "\n".join(lines)


MedSimplify terms loaded: 3100


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## CELL 8 — Stratified Sample

In [10]:
def add_length_bucket(df: pd.DataFrame, text_col: str = "complex") -> pd.DataFrame:
    df = df.copy()

    def bucket(text: object) -> str:
        n_words = safe_word_count(text)
        if n_words < 12:
            return "short"
        if n_words < 25:
            return "medium"
        return "long"

    df["len_bucket"] = df[text_col].apply(bucket)
    return df


def add_strata_column(
    df: pd.DataFrame,
    text_col: str = "complex",
    label_col: str = "label",
    use_label_and_length: bool = True,
) -> pd.DataFrame:
    df = add_length_bucket(df, text_col=text_col)

    if use_label_and_length and label_col in df.columns:
        df["strata"] = df[label_col].fillna("unknown").astype(str) + "__" + df["len_bucket"]
    elif label_col in df.columns:
        df["strata"] = df[label_col].fillna("unknown").astype(str)
    else:
        df["strata"] = df["len_bucket"]

    return df


def allocate_stratified_counts(counts: pd.Series, n_samples: int) -> Dict[str, int]:
    """
    Proportional stratified allocation that preserves the requested sample size
    whenever n_samples <= total rows. It does not collapse to one row per stratum.
    """
    counts = counts[counts > 0].astype(int)

    if n_samples >= counts.sum():
        return counts.to_dict()

    ideal = counts / counts.sum() * n_samples
    base = np.floor(ideal).astype(int)

    remaining = n_samples - int(base.sum())

    
    remainders = (ideal - base).sort_values(ascending=False)
    for stratum in remainders.index:
        if remaining <= 0:
            break
        if base[stratum] < counts[stratum]:
            base[stratum] += 1
            remaining -= 1

    if remaining > 0:
        for stratum in counts.sort_values(ascending=False).index:
            if remaining <= 0:
                break
            available = counts[stratum] - base[stratum]
            if available <= 0:
                continue
            add_k = min(available, remaining)
            base[stratum] += add_k
            remaining -= add_k

    
    while int(base.sum()) > n_samples:
        stratum = base[base > 0].idxmax()
        base[stratum] -= 1

    return base.astype(int).to_dict()


def stratified_sample(
    df: pd.DataFrame,
    n_samples: int = SAMPLE_SIZE,
    random_state: int = SEED,
    text_col: str = "complex",
    label_col: str = "label",
    use_label_and_length: bool = STRATIFY_BY_LABEL_AND_LENGTH,
) -> pd.DataFrame:
    if n_samples <= 0:
        raise ValueError("n_samples must be positive")

    df = add_strata_column(
        df,
        text_col=text_col,
        label_col=label_col,
        use_label_and_length=use_label_and_length,
    )

    n_samples = min(n_samples, len(df))
    counts = df["strata"].value_counts()
    allocation = allocate_stratified_counts(counts, n_samples)

    sampled_parts = []
    for stratum, k in allocation.items():
        if k <= 0:
            continue
        part = df[df["strata"] == stratum]
        sampled_parts.append(part.sample(n=k, random_state=random_state))

    sampled = pd.concat(sampled_parts, axis=0)
    sampled = sampled.sample(frac=1, random_state=random_state).reset_index(drop=True)

    if len(sampled) != n_samples:
        raise RuntimeError(f"Expected {n_samples} sampled rows, got {len(sampled)}")

    return sampled


def compare_distribution(full_df: pd.DataFrame, sample_df: pd.DataFrame, col: str) -> pd.DataFrame:
    full_dist = full_df[col].value_counts(normalize=True).rename("full_ratio")
    sample_dist = sample_df[col].value_counts(normalize=True).rename("sample_ratio")
    report = pd.concat([full_dist, sample_dist], axis=1).fillna(0)
    report["diff"] = report["sample_ratio"] - report["full_ratio"]
    return report.sort_values("full_ratio", ascending=False)


valid_subset = stratified_sample(valid_df, n_samples=SAMPLE_SIZE, random_state=SEED)
train_subset = stratified_sample(train_df, n_samples=min(SAMPLE_SIZE, len(train_df)), random_state=SEED)

print("valid_subset shape:", valid_subset.shape)
print("train_subset shape:", train_subset.shape)

if "label" in valid_df.columns:
    display(compare_distribution(valid_df, valid_subset, "label"))

display(valid_subset[["row_id", "complex", "simple", "label", "strata"]].head(5))


valid_subset shape: (100, 13)
train_subset shape: (100, 13)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,full_ratio,sample_ratio,diff
label,,,
rephrase,0.446671,0.45,0.003329
delete,0.348851,0.35,0.001149
ignore,0.098998,0.09,-0.008998
merge,0.037714,0.05,0.012286
split,0.034178,0.03,-0.004178
none,0.033589,0.03,-0.003589


,row_id,complex,simple,label,strata
0,426,About one-third of the studies were in urban/p...,['About one-third of the studies were in urban...,ignore,ignore__medium
1,1361,Washing of umbilical cord with soap and water ...,[],delete,delete__medium
2,603,We included 14 studies with 923 participants i...,"['The search, updated in May 2018, identified ...",rephrase,rephrase__short
3,1371,There is insufficient evidence to support the ...,[],delete,delete__long
4,1380,Quality of evidence There is moderate to very ...,[],delete,delete__long


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## CELL 9 — Planner Prompt

In [11]:
PLANNER_SYSTEM = """
You are a sentence simplification planner.
Return only strict JSON.
""".strip()


def parse_json_object(text: str) -> dict:
    text = normalize_text(text)
    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if match:
        text = match.group(0)
    return json.loads(text)


def semantic_planner(sentence: str, next_sentence: str = "") -> Tuple[dict, bool, bool]:
    prompt = f"""
Analyze the sentence and return STRICT JSON only.

Return exactly these keys:
{{
  "is_already_simple": boolean,
  "contains_technical_terms": boolean,
  "is_structurally_complex": boolean,
  "has_secondary_statistical_detail": boolean,
  "requires_abbreviation_expansion": boolean,
  "depends_on_next_sentence": boolean,
  "redundant_given_next_sentence": boolean,
  "reason": "short string"
}}

Definitions:
- is_already_simple: true only if the sentence is already easy for a general reader.
- contains_technical_terms: true if the sentence contains medical or scientific jargon, abbreviations, or difficult domain terms.
- is_structurally_complex: true if the sentence has multiple clauses or would benefit from splitting.
- has_secondary_statistical_detail: true if it mainly contains numerical/statistical detail that may be secondary.
- requires_abbreviation_expansion: true if an abbreviation should be expanded or clarified.
- depends_on_next_sentence: true if this sentence cannot be simplified safely without the next sentence.
- redundant_given_next_sentence: true if the next sentence repeats the same meaning more clearly.

Sentence:
{sentence}

Next sentence:
{next_sentence}
""".strip()

    raw = local_chat_call(
        prompt=prompt,
        system_message=PLANNER_SYSTEM,
        max_tokens=PLANNER_MAX_TOKENS,
        temperature=PLANNER_TEMPERATURE,
    )

    planner_json_ok = True
    used_default = False

    try:
        plan = parse_json_object(raw)

        required_keys = [
            "is_already_simple",
            "contains_technical_terms",
            "is_structurally_complex",
            "has_secondary_statistical_detail",
            "requires_abbreviation_expansion",
            "depends_on_next_sentence",
            "redundant_given_next_sentence",
            "reason",
        ]

        for key in required_keys:
            if key not in plan:
                raise ValueError(f"Missing planner key: {key}")

        for key in required_keys[:-1]:
            if isinstance(plan[key], str):
                plan[key] = plan[key].strip().lower() == "true"
            else:
                plan[key] = bool(plan[key])

        plan["reason"] = str(plan["reason"])

    except Exception as parse_error:
        planner_json_ok = False
        used_default = True
        plan = {
            "is_already_simple": False,
            "contains_technical_terms": False,
            "is_structurally_complex": False,
            "has_secondary_statistical_detail": False,
            "requires_abbreviation_expansion": False,
            "depends_on_next_sentence": False,
            "redundant_given_next_sentence": False,
            "reason": f"default_fallback_parse_error: {parse_error}; raw={raw[:200]}",
        }

    return plan, planner_json_ok, used_default

## CELL 10 — Rule-Based Strategy Decision

In [12]:
def count_commas(sentence: str) -> int:
    return str(sentence).count(",")


def is_long_sentence(sentence: str, threshold: int = 22) -> bool:
    return safe_word_count(sentence) >= threshold


def has_clause_markers(sentence: str) -> bool:
    s = f" {str(sentence).lower()} "
    markers = [
        " although ", " however ", " whereas ", " while ",
        " because ", " therefore ", " which ", " that ",
        " and ", " but ", " or ", " with ", " without ",
        " after ", " before "
    ]
    return any(marker in s for marker in markers)


def has_numeric_detail(sentence: str) -> bool:
    s = str(sentence).lower()

    numeric_patterns = [
        r"\d+",
        r"\d+\.\d+",
        r"\d+%",
        r"\d+\s*to\s*\d+",
        r"\d+\s*-\s*\d+",
        r"\(\s*\d+",
    ]

    statistical_terms = [
        "confidence interval", "ci", "risk ratio", "odds ratio",
        "hazard ratio", "mean difference", "standard deviation",
        "p value", "p-value", "range", "median", "interquartile",
        "iqr", "participants", "records", "studies", "trials",
        "sample size",
    ]

    return any(re.search(pattern, s) for pattern in numeric_patterns) or any(term in s for term in statistical_terms)


def has_secondary_evidence_wording(sentence: str) -> bool:
    s = str(sentence).lower()
    phrases = [
        "low-certainty evidence",
        "very low-certainty evidence",
        "low quality evidence",
        "very low quality evidence",
        "certainty of the evidence",
        "quality of the evidence",
        "insufficient evidence",
        "no clear evidence",
        "did not report",
        "not reported",
        "not possible to determine",
        "no studies reported",
        "secondary outcomes",
        "remaining evidence",
    ]
    return any(phrase in s for phrase in phrases)


def is_mostly_statistical_detail(sentence: str) -> bool:
    s = str(sentence).lower()
    n_words = safe_word_count(s)
    if n_words == 0:
        return False

    number_count = sum(ch.isdigit() for ch in s)
    statistical_keywords = [
        "confidence", "interval", "range", "median",
        "mean", "ratio", "risk", "odds",
        "participants", "studies", "trials",
        "sample", "p-value",
    ]
    keyword_hits = sum(1 for keyword in statistical_keywords if keyword in s)

    return has_numeric_detail(s) and (keyword_hits >= 2 or number_count >= 4 or n_words <= 18)



def has_informative_numeric_context(sentence: str) -> bool:
    """
    Detects numeric/statistical sentences that still contain important
    scientific information and should not be deleted aggressively.
    """
    s = str(sentence).lower()

    informative_terms = [
        "trial", "trials",
        "study", "studies",
        "participants", "patients", "infants", "women", "children",
        "follow-up", "follow up",
        "weeks", "months", "years",
        "assessed", "included", "compared", "reported",
        "outcome", "outcomes",
        "intervention", "interventions",
        "effect", "effects",
        "evidence",
        "sample", "median", "range",
    ]

    return any(term in s for term in informative_terms)


def decide_strategy(plan: dict, verified_terms: Optional[Dict[str, str]] = None, sentence: str = "") -> str:
    verified_terms = verified_terms or {}
    sentence = normalize_text(sentence)

    n_words = safe_word_count(sentence)

    long_sentence = n_words >= 26
    very_long_sentence = n_words >= 36
    comma_heavy = count_commas(sentence) >= 3
    clause_heavy = has_clause_markers(sentence)

    numeric_detail = has_numeric_detail(sentence)
    secondary_evidence = has_secondary_evidence_wording(sentence)
    mostly_stats = is_mostly_statistical_detail(sentence)
    informative_numeric = has_informative_numeric_context(sentence)

    has_terms = bool(verified_terms) or plan.get("contains_technical_terms", False)
    needs_abbrev = plan.get("requires_abbreviation_expansion", False)
    structurally_complex = plan.get("is_structurally_complex", False)

    
    if plan.get("is_already_simple", False) and not needs_abbrev:
        return "KEEP_AS_IS"


    if (
        n_words <= 18
        and not needs_abbrev
        and not structurally_complex
        and count_commas(sentence) <= 1
        and not secondary_evidence
        and not has_terms
    ):
        return "KEEP_AS_IS"

    # 3) Avoid deleting informative scientific/statistical sentences.
    # Numeric details are not always secondary in Cochrane-style text.
    if (
        mostly_stats
        and not informative_numeric
        and not has_terms
        and n_words <= 16
    ):
        return "DELETE_OR_OMIT"

    if (
        plan.get("has_secondary_statistical_detail", False)
        and not plan.get("depends_on_next_sentence", False)
        and not informative_numeric
        and not has_terms
        and n_words <= 18
    ):
        return "DELETE_OR_OMIT"

    # 4) Evidence wording usually carries meaning.
    # Rephrase it instead of deleting it.
    if secondary_evidence:
        if n_words <= 10 and not informative_numeric:
            return "DELETE_OR_OMIT"
        return "REPHRASE"

    # 5) Long numeric sentences need simplification, not deletion.
    if numeric_detail and very_long_sentence:
        return "SPLIT_AND_SIMPLIFY"

    # 6) Terminology handling.
    # Avoid standalone SUBSTITUTE unless the sentence is short and the terms are central.
    # In most cases, rephrase is safer than direct term substitution.
    if has_terms:
        if long_sentence or comma_heavy or structurally_complex:
            return "SPLIT_AND_SIMPLIFY"
        return "REPHRASE"

    # 7) Structural complexity.
    # Prefer SPLIT_AND_SIMPLIFY for longer complex sentences.
    # Use REPHRASE for medium complexity to avoid unnecessary splitting.
    if structurally_complex or comma_heavy or (long_sentence and clause_heavy):
        if n_words >= 24:
            return "SPLIT_AND_SIMPLIFY"
        return "REPHRASE"

    return "REPHRASE"

## CELL 11 — Execution Prompts

In [13]:
SHARED_RULES = """
Critical rules:
- Preserve the original scientific meaning.
- Do not add information that is not explicitly supported by the sentence.
- Do not invent causes, effects, or explanations.
- Do not over-explain.
- Keep the output short, natural, and clear.
- Output only the final simplified text.
""".strip()

REPHRASE_PROMPT = """
Task: Rewrite this scientific or medical sentence in plainer English for a general reader.

Instructions:
- Preserve the original meaning.
- Do not add new facts.
- Do not remove important clinical or scientific information.
- If terminology substitution was already applied, keep the clearer wording unless it would change the meaning.
- If the sentence contains confidence intervals, odds ratios, I2 values, or trial counts, keep only the main conclusion unless these details are essential.
- Prefer a shorter sentence than the source.
- Keep the uncertainty level, such as uncertain, unclear, or low-certainty evidence.
- When a numerical range includes values both below and above the baseline, do not describe it as an increase or decrease. Use neutral wording such as "might be between X and Y".
- Preserve uncertainty. Do not turn an uncertain comparison into a definite improvement or reduction.

{rag_context}

{shared_rules}

Sentence:
{sentence}

Simplified:
""".strip()

SPLIT_AND_SIMPLIFY_PROMPT = """
Task: Simplify this scientific or medical sentence.

Instructions:
- Split the sentence only if it improves readability.
- Use shorter and clearer wording.
- Preserve the original scientific meaning.
- Do not add new facts.
- Do not remove important clinical or scientific information.
- If terminology substitution was already applied, keep the clearer wording unless it would change the meaning.
- If the sentence contains confidence intervals, odds ratios, I2 values, or trial counts, keep only the main conclusion unless these details are essential.
- Avoid unnecessary expansion.
- The output may be one or two short sentences.
- When a numerical range includes values both below and above the baseline, do not describe it as an increase or decrease. Use neutral wording such as "might be between X and Y".
- Preserve uncertainty. Do not turn an uncertain comparison into a definite improvement or reduction.

{rag_context}

{shared_rules}

Sentence:
{sentence}

Simplified:
""".strip()

DELETE_OR_OMIT_PROMPT = """
Task: Decide whether this scientific or medical sentence can be safely omitted.

Rules:
- Output exactly OMIT only if the sentence is clearly redundant, administrative, or contains secondary information that can be safely omitted.
- Do NOT omit the sentence if it contains important information about trials, studies, participants, follow-up, outcomes, effects, evidence, or comparisons.
- If the sentence contains useful scientific information, rewrite it briefly in simpler English instead of omitting it.
- Preserve the original meaning.
- Do not add new facts.
- Output only OMIT or the final simplified sentence.

Sentence:
{sentence}

Result:
""".strip()

SUBSTITUTE_PROMPT = """
Task: Rewrite this scientific or medical sentence in plainer English for a general reader.
Focus especially on replacing difficult medical terms with simpler wording using the provided terminology hints.
Then lightly rephrase the full sentence so it sounds natural and clear.

Critical rules:
- Replace a difficult medical term only when the simpler wording is medically safe and accurate.
- If no safe simpler wording exists, keep the original term.
- Do not guess meanings.
- Do not change the affected organ, body system, condition type, population, intervention, or outcome.
- Do not turn a diagnosis into a generic disease unless the meaning remains exact.
- Prefer standard plain-English equivalents when they are provided in the terminology hints.
- Preserve the original meaning exactly.
- Do not add information that is not explicitly supported by the sentence.
- Do not invent causes, effects, or explanations.
- Keep the sentence natural and concise.
- Output only the final sentence.
-If two terms are separated by "/", rewrite them using "or" instead of keeping the slash.

Examples of safe plain-English equivalents:
- myocardial infarction -> heart attack
- tachycardia -> rapid heart rate
- hypotension -> low blood pressure
- hypertension -> high blood pressure
- cerebrovascular accident -> stroke
- glucocorticoids -> steroid medicines

{rag_context}

Sentence:
{sentence}

Simplified:
""".strip()


def call_generation_prompt(prompt: str) -> str:
    return local_chat_call(
        prompt=prompt,
        system_message="You are a careful medical text simplification expert.",
        max_tokens=MAX_NEW_TOKENS,
        temperature=EXECUTOR_TEMPERATURE,
    )


def validate_output(original: str, output: str, strategy: str) -> str:
    original = normalize_text(original)
    output = normalize_text(output)
    strategy = str(strategy).upper()

    if strategy == "KEEP_AS_IS":
        return original

    if strategy == "DELETE_OR_OMIT":
        if output.upper() == "OMIT":
            return ""
        return output if output else original

    return output if output else original


def has_real_rag_terms(rag_context: str) -> bool:
    if not rag_context:
        return False
    return any(line.strip().startswith("- ") and (":" in line or "->" in line) for line in str(rag_context).splitlines())


def run_rephrase(sentence: str, rag_context: str = "") -> str:
    prompt = REPHRASE_PROMPT.format(
        sentence=sentence,
        rag_context=rag_context,
        shared_rules=SHARED_RULES,
    )
    return call_generation_prompt(prompt)


def run_split_and_simplify(sentence: str, rag_context: str = "") -> str:
    prompt = SPLIT_AND_SIMPLIFY_PROMPT.format(
        sentence=sentence,
        rag_context=rag_context,
        shared_rules=SHARED_RULES,
    )
    return call_generation_prompt(prompt)


def run_delete_or_omit(sentence: str) -> str:
    prompt = DELETE_OR_OMIT_PROMPT.format(sentence=sentence)
    return call_generation_prompt(prompt)


def run_substitute(sentence: str, rag_context: str = "") -> str:
    prompt = SUBSTITUTE_PROMPT.format(
        sentence=sentence,
        rag_context=rag_context,
    )
    return call_generation_prompt(prompt)


def execute_strategy(
    sentence: str,
    strategy: str,
    rag_context: str = "",
    found_terms: Optional[Dict[str, str]] = None,
) -> Tuple[str, str, str, bool]:
    """
    Execute the full simplification flow:

    1. If real medical terminology is detected, run an LLM-based substitution step first.
    2. Apply the selected final strategy: keep, rephrase, split, or delete/omit.

    Returns:
        final_output, execution_mode, sentence_after_substitution, substitution_applied
    """
    strategy = str(strategy or "").strip().upper()
    original_sentence = normalize_text(sentence)
    working_sentence = original_sentence
    sentence_after_substitution = original_sentence
    substitution_applied = False
    has_terms = has_real_rag_terms(rag_context)

    if has_terms:
        sub_raw = run_substitute(working_sentence, rag_context=rag_context)
        sub_clean = validate_output(working_sentence, sub_raw, "SUBSTITUTE")

        if sub_clean and sub_clean.strip().lower() != working_sentence.strip().lower():
            working_sentence = normalize_text(sub_clean)
            sentence_after_substitution = working_sentence
            substitution_applied = True

    if strategy == "KEEP_AS_IS":
        mode = "keep_after_substitution" if substitution_applied else ("keep_with_terms" if has_terms else "keep")
        return working_sentence, mode, sentence_after_substitution, substitution_applied

    if strategy == "DELETE_OR_OMIT":
        raw = run_delete_or_omit(working_sentence)
        final = validate_output(working_sentence, raw, strategy)
        mode = "delete_or_omit_after_substitution" if substitution_applied else ("delete_or_omit_with_terms" if has_terms else "delete_or_omit")
        return final, mode, sentence_after_substitution, substitution_applied

    if strategy in {"SPLIT", "SPLIT_AND_SIMPLIFY"}:
        raw = run_split_and_simplify(working_sentence, rag_context=rag_context)
        final = validate_output(working_sentence, raw, "SPLIT_AND_SIMPLIFY")
        mode = "split_after_substitution" if substitution_applied else ("split_and_simplify_with_terms" if has_terms else "split_and_simplify")
        return final, mode, sentence_after_substitution, substitution_applied

    raw = run_rephrase(working_sentence, rag_context=rag_context)
    final = validate_output(working_sentence, raw, "REPHRASE")
    mode = "rephrase_after_substitution" if substitution_applied else ("rephrase_with_terms" if has_terms else "rephrase")
    return final, mode, sentence_after_substitution, substitution_applied


## CELL 12 — Checkpoint Helpers

In [14]:
def load_done_ids(csv_path: Path, id_col: str = "row_id") -> Tuple[set, pd.DataFrame]:
    if csv_path.exists():
        df = pd.read_csv(csv_path)
        if id_col in df.columns:
            return set(df[id_col].tolist()), df
        return set(), df
    return set(), pd.DataFrame()


def append_row_to_csv(row_dict: dict, csv_path: Path) -> None:
    row_df = pd.DataFrame([row_dict])
    if csv_path.exists():
        row_df.to_csv(csv_path, mode="a", header=False, index=False)
    else:
        row_df.to_csv(csv_path, index=False)


def prepare_split_dir(split_name: str) -> Path:
    split_dir = RUN_DIR / split_name
    split_dir.mkdir(parents=True, exist_ok=True)
    return split_dir

## CELL 13 — Planner Stage

In [15]:
def build_next_sentence_lookup(
    context_df: Optional[pd.DataFrame],
    text_col: str = "complex",
) -> Dict[Tuple[str, str], str]:
    """
    Build a next-sentence lookup using paragraph and sentence identifiers.
    This avoids using sampled row order as if it were the original sentence order.
    """
    if context_df is None:
        return {}

    required_cols = {"para_id", "sent_id", text_col}
    if not required_cols.issubset(set(context_df.columns)):
        return {}

    tmp = context_df.copy()
    tmp["_para_key"] = tmp["para_id"].astype(str)
    tmp["_sent_key"] = tmp["sent_id"].astype(str)
    tmp["_sent_num"] = pd.to_numeric(tmp["sent_id"], errors="coerce")

    lookup: Dict[Tuple[str, str], str] = {}
    for _, group in tmp.groupby("_para_key", sort=False):
        group = group.sort_values(["_sent_num", "_sent_key"], na_position="last").reset_index(drop=True)
        for idx in range(len(group)):
            key = (str(group.loc[idx, "_para_key"]), str(group.loc[idx, "_sent_key"]))
            next_sentence = normalize_text(group.loc[idx + 1, text_col]) if idx + 1 < len(group) else ""
            lookup[key] = next_sentence

    return lookup


def get_reliable_next_sentence(row: pd.Series, next_sentence_lookup: Dict[Tuple[str, str], str]) -> str:
    """Return the true next sentence when paragraph/sentence IDs are available; otherwise return empty context."""
    if not next_sentence_lookup:
        return ""
    if "para_id" not in row or "sent_id" not in row:
        return ""
    key = (str(row["para_id"]), str(row["sent_id"]))
    return normalize_text(next_sentence_lookup.get(key, ""))


def run_planner_stage(
    df: pd.DataFrame,
    split_name: str,
    text_col: str = "complex",
    reset_first: bool = False,
    context_df: Optional[pd.DataFrame] = None,
) -> pd.DataFrame:
    split_dir = prepare_split_dir(split_name)
    planner_csv = split_dir / "planner_stage.csv"

    if reset_first and planner_csv.exists():
        planner_csv.unlink()

    done_ids, _ = load_done_ids(planner_csv)
    total = len(df)
    next_sentence_lookup = build_next_sentence_lookup(context_df, text_col=text_col)

    for i, row in tqdm(df.iterrows(), total=total, desc=f"{split_name} planner"):
        row_id = int(row["row_id"]) if "row_id" in df.columns else int(i)

        if row_id in done_ids:
            continue

        original_sentence = normalize_text(row[text_col])
        next_sentence = get_reliable_next_sentence(row, next_sentence_lookup)

        try:
            found_terms = extract_terms(original_sentence)
            terminology_context = build_rag_context(found_terms)

            # Strategy selection remains based on the original sentence.
            # The substitution step is applied later during execution.
            plan, planner_json_ok, used_default = semantic_planner(
                original_sentence,
                next_sentence,
            )
            strategy = decide_strategy(plan, found_terms, original_sentence)

            out_row = {
                "row_id": row_id,
                "complex": original_sentence,
                "original_complex": original_sentence,
                "substituted_sentence": original_sentence,
                "simple": normalize_text(row.get("simple", "")),
                "label": normalize_text(row.get("label", "")),
                "next_sentence": next_sentence,
                "rag_terms": ", ".join(found_terms.keys()) if found_terms else "",
                "rag_terms_count": len(found_terms),
                "applied_substitutions": "{}",
                "substitution_applied": False,
                "terminology_context": terminology_context,
                "plan_json": json.dumps(plan, ensure_ascii=False),
                "planner_json_ok": planner_json_ok,
                "used_default": used_default,
                "strategy": strategy,
                "status": "planned",
            }
            append_row_to_csv(out_row, planner_csv)
            time.sleep(REQUEST_SLEEP_SEC)

        except Exception as error:
            error_row = {
                "row_id": row_id,
                "complex": original_sentence,
                "original_complex": original_sentence,
                "substituted_sentence": original_sentence,
                "simple": normalize_text(row.get("simple", "")),
                "label": normalize_text(row.get("label", "")),
                "next_sentence": next_sentence,
                "rag_terms": "",
                "rag_terms_count": 0,
                "applied_substitutions": "{}",
                "substitution_applied": False,
                "terminology_context": "",
                "plan_json": "",
                "planner_json_ok": False,
                "used_default": True,
                "strategy": "",
                "status": f"planner_error: {error}",
            }
            append_row_to_csv(error_row, planner_csv)
            print(f"Planner stopped at row_id={row_id}: {error}")
            break

    return pd.read_csv(planner_csv) if planner_csv.exists() else pd.DataFrame()


## CELL 14 — Executor Stage

In [16]:
def terms_from_string(rag_terms: object) -> Dict[str, str]:
    terms: Dict[str, str] = {}
    text = normalize_text(rag_terms)

    if not text:
        return terms

    for term in text.split(","):
        term = normalize_term_text(term)
        if term in med_lookup:
            terms[term] = med_lookup[term]
        elif term in SAFE_TERM_REPLACEMENTS:
            terms[term] = SAFE_TERM_REPLACEMENTS[term]

    return terms


def parse_json_dict(value: object) -> Dict[str, str]:
    """Safely parse a JSON dictionary stored in a CSV cell."""
    text = normalize_text(value)
    if not text:
        return {}
    try:
        parsed = json.loads(text)
        return parsed if isinstance(parsed, dict) else {}
    except Exception:
        return {}


def run_executor_stage(
    split_name: str,
    reset_first: bool = False,
) -> pd.DataFrame:
    split_dir = prepare_split_dir(split_name)
    planner_csv = split_dir / "planner_stage.csv"
    executor_csv = split_dir / "executor_stage.csv"

    if not planner_csv.exists():
        raise FileNotFoundError(f"Planner file not found: {planner_csv}")

    if reset_first and executor_csv.exists():
        executor_csv.unlink()

    planner_df = pd.read_csv(planner_csv)
    planner_df = planner_df[planner_df["status"].astype(str).str.startswith("planned")].copy()

    done_ids, _ = load_done_ids(executor_csv)
    total = len(planner_df)

    for _, row in tqdm(planner_df.iterrows(), total=total, desc=f"{split_name} executor"):
        row_id = int(row["row_id"])

        if row_id in done_ids:
            continue

        original_sentence = normalize_text(row.get("original_complex", row.get("complex", "")))
        strategy = normalize_text(row["strategy"])

        try:
            found_terms = terms_from_string(row.get("rag_terms", ""))
            rag_context = build_rag_context(found_terms)

            output, execution_mode, substituted_sentence, substitution_applied = execute_strategy(
                original_sentence,
                strategy,
                rag_context=rag_context,
                found_terms=found_terms,
            )

            substitution_details = {
                "method": "llm_substitution",
                "terms": list(found_terms.keys()),
            } if substitution_applied else {}

            out_row = {
                "row_id": row_id,
                "complex": original_sentence,
                "original_complex": original_sentence,
                "substituted_sentence": normalize_text(substituted_sentence),
                "simple": normalize_text(row.get("simple", "")),
                "label": normalize_text(row.get("label", "")),
                "strategy": strategy,
                "execution_mode": execution_mode,
                "rag_terms": normalize_text(row.get("rag_terms", "")),
                "rag_terms_count": int(row.get("rag_terms_count", 0)),
                "applied_substitutions": json.dumps(substitution_details, ensure_ascii=False),
                "substitution_applied": bool(substitution_applied),
                "terminology_context": rag_context,
                "planner_json_ok": bool(row.get("planner_json_ok", False)),
                "used_default": bool(row.get("used_default", False)),
                "output": normalize_text(output),
                "output_changed": normalize_text(output).lower() != original_sentence.lower(),
                "substitution_changed_input": normalize_text(substituted_sentence).lower() != original_sentence.lower(),
                "status": "executed",
            }
            append_row_to_csv(out_row, executor_csv)
            time.sleep(REQUEST_SLEEP_SEC)

        except Exception as error:
            error_row = {
                "row_id": row_id,
                "complex": original_sentence,
                "original_complex": original_sentence,
                "substituted_sentence": original_sentence,
                "simple": normalize_text(row.get("simple", "")),
                "label": normalize_text(row.get("label", "")),
                "strategy": strategy,
                "execution_mode": "",
                "rag_terms": normalize_text(row.get("rag_terms", "")),
                "rag_terms_count": int(row.get("rag_terms_count", 0)),
                "applied_substitutions": "{}",
                "substitution_applied": False,
                "terminology_context": "",
                "planner_json_ok": bool(row.get("planner_json_ok", False)),
                "used_default": bool(row.get("used_default", False)),
                "output": "",
                "output_changed": False,
                "substitution_changed_input": False,
                "status": f"executor_error: {error}",
            }
            append_row_to_csv(error_row, executor_csv)
            print(f"Executor stopped at row_id={row_id}: {error}")
            break

    return pd.read_csv(executor_csv) if executor_csv.exists() else pd.DataFrame()


## CELL 17 — Evaluation Helpers

In [18]:
import ast


def parse_reference_list(value: object) -> List[str]:
    """
    Converts the dataset simple column into a list of references.
    Handles plain strings, Python-list strings such as "['...']", and empty lists "[]".
    """
    if isinstance(value, list):
        refs = [normalize_text(v) for v in value if normalize_text(v)]
        return refs if refs else [""]

    text = "" if pd.isna(value) else str(value).strip()
    if not text:
        return [""]

    if text.startswith("[") and text.endswith("]"):
        try:
            parsed = ast.literal_eval(text)
            if isinstance(parsed, list):
                refs = [normalize_text(v) for v in parsed if normalize_text(v)]
                return refs if refs else [""]
        except Exception:
            pass

    return [normalize_text(text)]


def first_reference(value: object) -> str:
    refs = parse_reference_list(value)
    return refs[0] if refs else ""


def try_load_sari_metric():
    try:
        import evaluate
        return evaluate.load("sari")
    except Exception as error:
        raise ImportError(
            "Could not load evaluate/sari. Install evaluation packages first: "
            "!pip install -q evaluate sacremoses sacrebleu textstat"
        ) from error


sari_metric = try_load_sari_metric()


def safe_bleu(prediction: str, reference: object) -> float:
    prediction = normalize_text(prediction)
    reference_text = first_reference(reference)

    if not prediction or not reference_text:
        return 0.0

    try:
        import sacrebleu
        return float(sacrebleu.sentence_bleu(prediction, [reference_text]).score)
    except Exception as error:
        print("BLEU error:", error)
        return 0.0


def safe_sari_batch(sources, predictions, references) -> float:
    try:
        ref_lists = [parse_reference_list(ref) for ref in references]
        result = sari_metric.compute(
            sources=[normalize_text(x) for x in sources],
            predictions=[normalize_text(x) for x in predictions],
            references=ref_lists,
        )
        return float(result["sari"])
    except Exception as error:
        print("SARI error:", error)
        return 0.0


def safe_sari_sentence(source: str, prediction: str, reference: object) -> float:
    return safe_sari_batch([source], [prediction], [reference])


def safe_mean(values) -> float:
    arr = pd.to_numeric(pd.Series(values), errors="coerce").to_numpy(dtype=float)
    valid = arr[~np.isnan(arr)]
    if len(valid) == 0:
        return 0.0
    return float(valid.mean())


def metric_text(value: float, digits: int = 4) -> str:
    if value is None or pd.isna(value):
        return "N/A"
    return f"{float(value):.{digits}f}"


def add_eval_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    for col in ["complex", "simple", "output"]:
        if col not in df.columns:
            df[col] = ""
        df[col] = df[col].fillna("").astype(str).apply(normalize_text)

    if "original_complex" in df.columns:
        df["source_for_metrics"] = df["original_complex"].fillna("").astype(str).apply(normalize_text)
        df.loc[df["source_for_metrics"] == "", "source_for_metrics"] = df["complex"]
    else:
        df["source_for_metrics"] = df["complex"]

    if "rag_terms_count" not in df.columns:
        df["rag_terms_count"] = 0
    df["rag_terms_count"] = df["rag_terms_count"].fillna(0).astype(int)

    df["reference_text"] = df["simple"].apply(first_reference)

    df["fkgl_src"] = df["source_for_metrics"].apply(safe_fkgl)
    df["fkgl_pred"] = df["output"].apply(safe_fkgl)
    df["fkgl_src_available"] = df["fkgl_src"].notna()
    df["fkgl_pred_available"] = df["fkgl_pred"].notna()
    df["fkgl_delta"] = df["fkgl_pred"] - df["fkgl_src"]

    df["compression"] = df.apply(
        lambda row: safe_word_count(row["output"]) / max(1, safe_word_count(row["source_for_metrics"])),
        axis=1,
    )
    df["bleu_sentence"] = df.apply(
        lambda row: safe_bleu(row["output"], row["simple"]),
        axis=1,
    )
    df["sari_sentence"] = df.apply(
        lambda row: safe_sari_sentence(row["source_for_metrics"], row["output"], row["simple"]),
        axis=1,
    )
    df["has_rag_terms"] = df["rag_terms_count"] > 0
    df["output_changed"] = df["source_for_metrics"].str.lower() != df["output"].str.lower()

    return df


def print_split_report(df: pd.DataFrame, title: str) -> pd.DataFrame:
    report_df = add_eval_columns(df)
    overall_sari = safe_sari_batch(report_df["source_for_metrics"], report_df["output"], report_df["simple"])

    print("=" * 80)
    print(f"{title} STRATEGY DISTRIBUTION")
    print(report_df["strategy"].value_counts(dropna=False))

    print("\n" + "=" * 80)
    print(f"{title} EXECUTION MODE DISTRIBUTION")
    print(report_df["execution_mode"].value_counts(dropna=False))

    print("\n" + "=" * 80)
    print(f"{title} EVALUATION REPORT")
    print("=" * 80)
    print(f"Total sentences    : {len(report_df)}")
    print(f"Overall SARI       : {metric_text(overall_sari, 4)}")
    print(f"Avg BLEU           : {metric_text(safe_mean(report_df['bleu_sentence']), 4)}")
    print(f"Avg FKGL source    : {metric_text(safe_mean(report_df['fkgl_src']), 2)}")
    print(f"Avg FKGL pred      : {metric_text(safe_mean(report_df['fkgl_pred']), 2)}")
    print(f"Avg FKGL delta     : {metric_text(safe_mean(report_df['fkgl_delta']), 2)}")
    print(f"Avg compression    : {metric_text(safe_mean(report_df['compression']), 3)}")
    print(f"Planner JSON OK    : {int(report_df['planner_json_ok'].sum())} / {len(report_df)}")
    print(f"Used default plan  : {int(report_df['used_default'].sum())} / {len(report_df)}")
    print(f"Output changed     : {int(report_df['output_changed'].sum())} / {len(report_df)}")
    print(f"With terminology   : {int(report_df['has_rag_terms'].sum())} / {len(report_df)}")
    if "substitution_applied" in report_df.columns:
        print(f"With substitution  : {int(report_df['substitution_applied'].fillna(False).astype(bool).sum())} / {len(report_df)}")
    print(f"FKGL unavailable   : {int(report_df['fkgl_pred'].isna().sum())} outputs")

    print("\nPer-strategy:")
    for strategy in report_df["strategy"].dropna().unique():
        sub = report_df[report_df["strategy"] == strategy]
        strategy_sari = safe_sari_batch(sub["source_for_metrics"], sub["output"], sub["simple"])
        print(
            f"- {strategy}: n={len(sub)}, "
            f"SARI={metric_text(strategy_sari, 4)}, "
            f"BLEU={metric_text(safe_mean(sub['bleu_sentence']), 4)}, "
            f"Compression={metric_text(safe_mean(sub['compression']), 3)}"
        )

    return report_df


## CELL 18 — Evaluate Completed Run

In [ ]:
executor_path = RUN_DIR / SAMPLE_SPLIT_NAME / "executor_stage.csv"

if not executor_path.exists():
    raise FileNotFoundError(f"Executor output not found: {executor_path}")

completed_df = pd.read_csv(executor_path)
report_df = print_split_report(completed_df, SAMPLE_SPLIT_NAME)

report_path = RUN_DIR / SAMPLE_SPLIT_NAME / "evaluation_report.csv"
report_df.to_csv(report_path, index=False)

print("Saved report:", report_path)

show_cols = ["complex", "simple", "output", "strategy", "execution_mode", "rag_terms", "sari_sentence"]
display(report_df.sort_values("sari_sentence", ascending=True).head(10)[show_cols])
display(report_df.sort_values("sari_sentence", ascending=False).head(10)[show_cols])


## CELL 19 —  Run on Valid & Test

In [21]:

valid_900_subset = stratified_sample(
    valid_df,
    n_samples=min(VALID_RUN_SIZE, len(valid_df)),
    random_state=SEED,
)

internal_test_800_subset = stratified_sample(
    test_internal_df,
    n_samples=min(INTERNAL_TEST_RUN_SIZE, len(test_internal_df)),
    random_state=SEED,
)

print("valid_subset:", valid_900_subset.shape)
print("internal_test_subset:", internal_test_800_subset.shape)

assert len(valid_900_subset) == min(VALID_RUN_SIZE, len(valid_df))
assert len(internal_test_800_subset) == min(INTERNAL_TEST_RUN_SIZE, len(test_internal_df))



valid_subset: (500, 13)
internal_test_subset: (500, 13)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## CELL 19A — Run Validation Sample 

In [23]:
valid_planner_df = run_planner_stage(
    valid_900_subset,
    split_name=VALID_RUN_SPLIT_NAME,
    context_df=valid_df,
    reset_first=RESET_BATCH_RUNS,
)

valid_executor_df = run_executor_stage(
    split_name=VALID_RUN_SPLIT_NAME,
    reset_first=RESET_BATCH_RUNS,
)

valid_report_df = print_split_report(valid_executor_df, VALID_RUN_SPLIT_NAME)
valid_report_path = RUN_DIR / VALID_RUN_SPLIT_NAME / "evaluation_report.csv"
valid_report_df.to_csv(valid_report_path, index=False)

print("Saved valid report:", valid_report_path)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


valid_900_stratified_qwen25_14b_awq_final_v1 planner:   0%|          | 0/500 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:653: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.d

valid_900_stratified_qwen25_14b_awq_final_v1 executor:   0%|          | 0/500 [00:00<?, ?it/s]

valid_900_stratified_qwen25_14b_awq_final_v1 STRATEGY DISTRIBUTION
strategy
SPLIT_AND_SIMPLIFY    260
REPHRASE              193
KEEP_AS_IS             47
Name: count, dtype: int64

valid_900_stratified_qwen25_14b_awq_final_v1 EXECUTION MODE DISTRIBUTION
execution_mode
split_after_substitution       172
rephrase_after_substitution    113
split_and_simplify              88
rephrase                        80
keep                            45
keep_after_substitution          2
Name: count, dtype: int64

valid_900_stratified_qwen25_14b_awq_final_v1 EVALUATION REPORT
Total sentences    : 500
Overall SARI       : 38.9789
Avg BLEU           : 9.2603
Avg FKGL source    : 12.64
Avg FKGL pred      : 11.33
Avg FKGL delta     : -1.31
Avg compression    : 0.868
Planner JSON OK    : 500 / 500
Used default plan  : 0 / 500
Output changed     : 455 / 500
With terminology   : 287 / 500
With substitution  : 287 / 500
FKGL unavailable   : 0 outputs

Per-strategy:
- KEEP_AS_IS: n=47, SARI=57.2795, BLEU=24.

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## CELL 19B — Run Test Sample 

In [ ]:
internal_test_planner_df = run_planner_stage(
    internal_test_800_subset,
    split_name=INTERNAL_TEST_RUN_SPLIT_NAME,
    context_df=test_internal_df,
    reset_first=RESET_BATCH_RUNS,
)

internal_test_executor_df = run_executor_stage(
    split_name=INTERNAL_TEST_RUN_SPLIT_NAME,
    reset_first=RESET_BATCH_RUNS,
)

internal_test_report_df = print_split_report(internal_test_executor_df, INTERNAL_TEST_RUN_SPLIT_NAME)
internal_test_report_path = RUN_DIR / INTERNAL_TEST_RUN_SPLIT_NAME / "evaluation_report.csv"
internal_test_report_df.to_csv(internal_test_report_path, index=False)

print("Saved internal test report:", internal_test_report_path)


## CELL 20 — Demo Interface



In [25]:
# ============================================================
# Standalone Gradio simplification interface with dataset examples
# ============================================================

import json
import gradio as gr


# ------------------------------------------------------------
# 1) Prepare examples from a dataframe
# ------------------------------------------------------------

INTERFACE_DF = valid_df.copy()

require_columns(INTERFACE_DF, ["complex", "simple"], "INTERFACE_DF")

INTERFACE_DF["complex"] = INTERFACE_DF["complex"].apply(normalize_text)
INTERFACE_DF["simple"] = INTERFACE_DF["simple"].apply(normalize_text)

INTERFACE_DF = INTERFACE_DF[
    (INTERFACE_DF["complex"].str.len() > 0) &
    (INTERFACE_DF["simple"].str.len() > 0)
].copy()

INTERFACE_DF = INTERFACE_DF.reset_index(drop=True)


def make_example_label(row_index: int, row: pd.Series) -> str:
    """
    Creates a short readable label for the dropdown without exposing dataset labels.
    """
    text = normalize_text(row.get("complex", ""))
    preview = text[:120] + "..." if len(text) > 120 else text
    return f"{row_index} | {preview}"


EXAMPLE_LABELS = [
    make_example_label(i, row)
    for i, row in INTERFACE_DF.iterrows()
]

LABEL_TO_INDEX = {
    label: i
    for i, label in enumerate(EXAMPLE_LABELS)
}


def get_next_sentence_from_interface_df(row_index: int) -> str:
    """
    Retrieves the next sentence using para_id and sent_id when available.
    Falls back to the next row only if identifiers are unavailable.
    """
    if row_index < 0 or row_index >= len(INTERFACE_DF):
        return ""

    row = INTERFACE_DF.iloc[row_index]

    if {"para_id", "sent_id"}.issubset(INTERFACE_DF.columns):
        para_id = row.get("para_id")
        sent_id = row.get("sent_id")

        try:
            next_sent_id = int(sent_id) + 1
            tmp_sent_id = pd.to_numeric(INTERFACE_DF["sent_id"], errors="coerce")
            candidates = INTERFACE_DF[
                (INTERFACE_DF["para_id"] == para_id) &
                (tmp_sent_id == next_sent_id)
            ]

            if len(candidates) > 0:
                return normalize_text(candidates.iloc[0]["complex"])
        except Exception:
            pass

    if row_index + 1 < len(INTERFACE_DF):
        return normalize_text(INTERFACE_DF.iloc[row_index + 1]["complex"])

    return ""


def load_example_from_dropdown(example_label: str):
    """
    Loads complex sentence, reference simplification, and optional next sentence.
    """
    if not example_label:
        return "", "", ""

    row_index = LABEL_TO_INDEX.get(example_label)
    if row_index is None:
        return "", "", ""

    row = INTERFACE_DF.iloc[row_index]

    complex_sentence = normalize_text(row.get("complex", ""))
    reference_sentence = normalize_text(row.get("simple", ""))
    next_sentence = get_next_sentence_from_interface_df(row_index)

    return complex_sentence, next_sentence, reference_sentence


# ------------------------------------------------------------
# 2) Single sentence pipeline
# ------------------------------------------------------------

def compute_single_sentence_metrics(source: str, prediction: str, reference: str = "") -> dict:
    """
    Computes sentence-level evaluation metrics for one example.

    FKGL and compression do not require a reference.
    BLEU and SARI require a reference simplification.
    """
    source = normalize_text(source)
    prediction = normalize_text(prediction)
    reference = normalize_text(reference)

    source_words = safe_word_count(source)
    prediction_words = safe_word_count(prediction)

    fkgl_source = safe_fkgl(source)
    fkgl_prediction = safe_fkgl(prediction)

    compression = prediction_words / source_words if source_words > 0 else None

    fkgl_delta = None
    if not pd.isna(fkgl_source) and not pd.isna(fkgl_prediction):
        fkgl_delta = fkgl_prediction - fkgl_source

    bleu = None
    sari = None

    if reference:
        bleu = safe_bleu(prediction, reference)
        sari = safe_sari_sentence(source, prediction, reference)

    return {
        "source_words": source_words,
        "prediction_words": prediction_words,
        "compression": compression,
        "fkgl_source": fkgl_source,
        "fkgl_prediction": fkgl_prediction,
        "fkgl_delta": fkgl_delta,
        "bleu": bleu,
        "sari": sari,
        "has_reference": bool(reference),
    }


def format_metric_value(value, digits=3):
    """
    Formats metric values for clean interface display.
    """
    if value is None:
        return "Reference not provided"

    try:
        if pd.isna(value):
            return "N/A"
    except Exception:
        pass

    if isinstance(value, float):
        return f"{value:.{digits}f}"

    return str(value)


def build_processing_summary(substitution_applied: bool, found_terms: dict) -> str:
    """
    Builds a user-facing processing summary without exposing internal strategy labels.
    """
    if substitution_applied:
        return "The system used terminology support before generating the simplified sentence."

    if found_terms:
        return "The system detected terminology and generated a simplified sentence."

    return "The system generated a simplified sentence directly."


def simplify_single_sentence(sentence: str, next_sentence: str = "", reference: str = "") -> dict:
    """
    Runs the same system pipeline on one manually entered sentence.
    This function does not read or write batch checkpoint files.
    """
    original_sentence = normalize_text(sentence)
    next_sentence = normalize_text(next_sentence)
    reference = normalize_text(reference)

    if not original_sentence:
        raise ValueError("Please enter a sentence first.")

    found_terms = extract_terms(original_sentence, max_terms=RAG_MAX_TERMS)
    terminology_context = build_rag_context(found_terms)

    plan, planner_json_ok, used_default = semantic_planner(
        original_sentence,
        next_sentence=next_sentence,
    )

    strategy = decide_strategy(
        plan=plan,
        verified_terms=found_terms,
        sentence=original_sentence,
    )

    output, execution_mode, substituted_sentence, substitution_applied = execute_strategy(
        original_sentence,
        strategy,
        rag_context=terminology_context,
        found_terms=found_terms,
    )

    output = normalize_text(output)

    substitution_details = {
        "method": "llm_substitution",
        "terms": list(found_terms.keys()),
    } if substitution_applied else {}

    metrics = compute_single_sentence_metrics(
        source=original_sentence,
        prediction=output,
        reference=reference,
    )

    processing_summary = build_processing_summary(
        substitution_applied=bool(substitution_applied),
        found_terms=found_terms,
    )

    return {
        "complex": original_sentence,
        "substituted_sentence": substituted_sentence,
        "next_sentence": next_sentence,
        "reference": reference,
        "output": output,
        "strategy": strategy,
        "execution_mode": execution_mode,
        "processing_summary": processing_summary,
        "terminology_terms": ", ".join(found_terms.keys()) if found_terms else "No terminology detected",
        "applied_substitutions": substitution_details,
        "applied_substitutions_text": json.dumps(substitution_details, ensure_ascii=False) if substitution_applied else "No LLM substitution applied",
        "substitution_applied": bool(substitution_applied),
        "terminology_context": terminology_context if terminology_context else "No terminology context used.",
        "plan_json": plan,
        "planner_json_ok": planner_json_ok,
        "used_default": used_default,
        "metrics": metrics,
    }


def simplify_for_interface(sentence: str, next_sentence: str, reference: str):
    """
    Wrapper function for Gradio.
    """
    try:
        result = simplify_single_sentence(
            sentence=sentence,
            next_sentence=next_sentence,
            reference=reference,
        )

        plan_pretty = json.dumps(
            result["plan_json"],
            indent=2,
            ensure_ascii=False,
        )

        metrics = result["metrics"]

        return (
            result["output"],
            result["processing_summary"],
            result["terminology_terms"],
            result["applied_substitutions_text"],
            result["terminology_context"],
            str(result["planner_json_ok"]),
            str(result["used_default"]),
            format_metric_value(metrics["sari"], digits=3),
            format_metric_value(metrics["bleu"], digits=3),
            format_metric_value(metrics["fkgl_source"], digits=3),
            format_metric_value(metrics["fkgl_prediction"], digits=3),
            format_metric_value(metrics["fkgl_delta"], digits=3),
            format_metric_value(metrics["compression"], digits=3),
            plan_pretty,
        )

    except Exception as error:
        error_message = f"Error: {str(error)}"

        return (
            error_message,
            "",
            "",
            "",
            "",
            "",
            "",
            "",
            "",
            "",
            "",
            "",
            "",
            "",
        )


# ------------------------------------------------------------
# 3) Interface design
# ------------------------------------------------------------

custom_css = """
.gradio-container {
    max-width: 1150px !important;
    margin: auto !important;
}

.main-title {
    text-align: center;
    margin-bottom: 8px;
}

.subtitle {
    text-align: center;
    color: #555;
    margin-bottom: 24px;
}

.output-box textarea {
    font-size: 15px !important;
}

.info-box textarea {
    font-size: 14px !important;
}
"""


with gr.Blocks(
    title="Scientific Text Simplification Demo",
    css=custom_css,
    theme=gr.themes.Soft(),
) as demo:

    gr.Markdown(
        """
        <h1 class="main-title">Scientific Text Simplification Demo</h1>
        <p class="subtitle">
        Select a dataset example or enter a sentence manually. The system detects terminology,
        generates a clearer version of the sentence, and reports sentence-level evaluation metrics.
        </p>
        """
    )

    example_dropdown = gr.Dropdown(
        label="Choose an Example from Validation Data",
        choices=EXAMPLE_LABELS,
        value=EXAMPLE_LABELS[0] if len(EXAMPLE_LABELS) > 0 else None,
        interactive=True,
    )

    load_example_button = gr.Button(
        "Load Selected Example",
        variant="secondary",
    )

    with gr.Row():
        with gr.Column(scale=1):
            sentence_input = gr.Textbox(
                label="Complex Scientific or Medical Sentence",
                placeholder="Enter a complex scientific or medical sentence here...",
                lines=6,
            )

            next_sentence_input = gr.Textbox(
                label="Optional Next Sentence",
                placeholder="Optional next sentence for context...",
                lines=3,
            )

            reference_input = gr.Textbox(
                label="Reference Simplification",
                placeholder="Reference simplification from the dataset. You can also write it manually.",
                lines=4,
            )

            with gr.Row():
                simplify_button = gr.Button("Simplify", variant="primary")
                clear_button = gr.Button("Clear")

        with gr.Column(scale=1):
            simplified_output = gr.Textbox(
                label="Simplified Output",
                lines=8,
                interactive=False,
                elem_classes=["output-box"],
            )

            processing_summary_output = gr.Textbox(
                label="Processing Summary",
                lines=2,
                interactive=False,
                elem_classes=["info-box"],
            )

    gr.Markdown("### Sentence-Level Evaluation Metrics")

    with gr.Row():
        sari_output = gr.Textbox(
            label="SARI",
            interactive=False,
            elem_classes=["info-box"],
        )

        bleu_output = gr.Textbox(
            label="BLEU",
            interactive=False,
            elem_classes=["info-box"],
        )

        compression_output = gr.Textbox(
            label="Compression Ratio",
            interactive=False,
            elem_classes=["info-box"],
        )

    with gr.Row():
        fkgl_source_output = gr.Textbox(
            label="Source FKGL",
            interactive=False,
            elem_classes=["info-box"],
        )

        fkgl_prediction_output = gr.Textbox(
            label="Prediction FKGL",
            interactive=False,
            elem_classes=["info-box"],
        )

        fkgl_delta_output = gr.Textbox(
            label="FKGL Delta",
            interactive=False,
            elem_classes=["info-box"],
        )

    gr.Markdown("### System Details")

    with gr.Row():
        terminology_terms_output = gr.Textbox(
            label="Terminology Terms Found",
            lines=2,
            interactive=False,
            elem_classes=["info-box"],
        )

        applied_substitutions_output = gr.Textbox(
            label="Terminology Handling",
            lines=2,
            interactive=False,
            elem_classes=["info-box"],
        )

    with gr.Row():
        planner_json_ok_output = gr.Textbox(
            label="Planner JSON OK",
            interactive=False,
            elem_classes=["info-box"],
        )

        used_default_output = gr.Textbox(
            label="Used Default Plan",
            interactive=False,
            elem_classes=["info-box"],
        )

    terminology_context_output = gr.Textbox(
        label="Terminology Context",
        lines=5,
        interactive=False,
        elem_classes=["info-box"],
    )

    planner_json_output = gr.Code(
        label="Planner JSON",
        language="json",
        lines=14,
    )

    load_example_button.click(
        fn=load_example_from_dropdown,
        inputs=[example_dropdown],
        outputs=[
            sentence_input,
            next_sentence_input,
            reference_input,
        ],
    )

    simplify_button.click(
        fn=simplify_for_interface,
        inputs=[
            sentence_input,
            next_sentence_input,
            reference_input,
        ],
        outputs=[
            simplified_output,
            processing_summary_output,
            terminology_terms_output,
            applied_substitutions_output,
            terminology_context_output,
            planner_json_ok_output,
            used_default_output,
            sari_output,
            bleu_output,
            fkgl_source_output,
            fkgl_prediction_output,
            fkgl_delta_output,
            compression_output,
            planner_json_output,
        ],
    )

    clear_button.click(
        fn=lambda: ("", "", "", "", "", "", "", "", "", "", "", "", "", "", "", "", ""),
        inputs=[],
        outputs=[
            sentence_input,
            next_sentence_input,
            reference_input,
            simplified_output,
            processing_summary_output,
            terminology_terms_output,
            applied_substitutions_output,
            terminology_context_output,
            planner_json_ok_output,
            used_default_output,
            sari_output,
            bleu_output,
            fkgl_source_output,
            fkgl_prediction_output,
            fkgl_delta_output,
            compression_output,
            planner_json_output,
        ],
    )


demo.launch(
    share=True,
    debug=False,
    show_error=True,
)

/tmp/ipykernel_144/1120969758.py:340: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_144/1120969758.py:340: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.no

* Running on local URL:  http://127.0.0.1:7862
* Running on public URL: https://f336c4108f7b8a8d38.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## CELL 21 — Prepare Official Submission JSON


In [ ]:
def save_submission_json(
    predictions_df: pd.DataFrame,
    output_json_path: Path,
    run_id: str = "SPU_DeepSimplify_task11_qwen25_14b_awq_strategy_v2",
) -> None:
    required = ["pair_id", "para_id", "sent_id", "output"]
    missing = [col for col in required if col not in predictions_df.columns]

    if missing:
        raise ValueError(f"Cannot create submission. Missing columns: {missing}")

    rows = []
    for _, row in predictions_df.iterrows():
        rows.append({
            "pair_id": row["pair_id"],
            "para_id": row["para_id"],
            "sent_id": row["sent_id"],
            "output": normalize_text(row["output"]),
        })

    payload = {
        "run_id": run_id,
        "predictions": rows,
    }

    output_json_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_json_path, "w", encoding="utf-8") as file:
        json.dump(payload, file, ensure_ascii=False, indent=2)

    print("Saved submission JSON:", output_json_path)


# Example:
# save_submission_json(official_predictions_df, OUTPUT_DIR / "SPU_DeepSimplify_task11_qwen25_14b_awq_strategy_v2.json")